# Easy human data: a hosted assembly hub

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cmdcolin/jbrowse-anywidget/blob/main/examples/08_hosted_assembly_hub.ipynb)

Wiring up a human genome by hand — sequence, refName aliases, cytobands, a gene-name search index — is the fiddly part. `fetch_hub` pulls all of it, already configured and CORS-enabled, from [genomes.jbrowse.org](https://genomes.jbrowse.org): pass a UCSC name (`hg38`, `hg19`, `mm10`) or a GenArk accession (`GCA_...`). It returns plain JSON you hand to the view.

In [ ]:
# Install only if not already available (e.g. in Colab). The GitHub install
# needs no JS toolchain — the built widget bundle is committed in the repo. A
# local editable install is used as-is. (Swap to `jbrowse-anywidget` once it's
# published to PyPI.)
try:
    import jbrowse_anywidget  # noqa: F401
except ImportError:
    %pip install -q "jbrowse-anywidget @ git+https://github.com/cmdcolin/jbrowse-anywidget" pandas numpy

# Colab requires this to render third-party (anywidget) widgets:
try:
    from google.colab import output

    output.enable_custom_widget_manager()
except ImportError:
    pass

## Pull hg38 and open it at a gene

The hub config carries a gene-name search index, so `location` accepts a symbol like `BRCA1`, not just a locstring.

In [ ]:
from jbrowse_anywidget import LinearGenomeView, fetch_hub

hg38 = fetch_hub("hg38")  # sequence + refName aliases + cytobands + search

view = LinearGenomeView(
    assembly=hg38["assemblies"][0],
    aggregate_text_search_adapters=hg38["aggregateTextSearchAdapters"],
    location="BRCA1",
)
view

## Add a hosted track

`hg38["tracks"]` is a catalog of ready-to-use hosted tracks. Pick one by id and hand it to `add_track` — it's just JSON, no special API.

In [ ]:
catalog = {t["trackId"]: t for t in hg38["tracks"]}
print(len(catalog), "hosted tracks, e.g.:", list(catalog)[:4])

view.add_track(catalog["hg38-ncbiRefSeqCurated"])

## Mix in your own data

Your own tracks drop in next to hosted ones. Because the hub assembly carries refName aliases, a file that names chromosomes `chr17` lines up with the reference automatically — no manual aliasing.

In [ ]:
view.add_track(
    {
        "type": "QuantitativeTrack",
        "trackId": "phyloP100way",
        "name": "phyloP100way",
        "assemblyNames": ["hg38"],
        "adapter": {
            "type": "BigWigAdapter",
            "uri": "https://hgdownload.cse.ucsc.edu/goldenpath/hg38/phyloP100way/hg38.phyloP100way.bw",
        },
    }
)